## TechMind — Exploración y Preparación del Dataset **StackExchange** (versión actualizada)

#### Equipo tejONEs

#### 04_exploracion_dataset_stackexchange.ipynb

💡 **Dataset**: [StackExchange — extracción a través de la API oficial](https://api.stackexchange.com/)

El proceso general que sigue este pipeline es:

- Extracción de preguntas mediante tags técnicos desde la API oficial de Stack Exchange.
- Recuperación de las respuestas aceptadas para usarlas como contenido, en vez del texto de las preguntas.
- Ampliación del alcance inicial de Mobile y Frontend hasta completar las siete categorías del proyecto.
- Normalización a las siete categorías y asignación del tipo de contenido `articulo`.
- Persistencia del dataset crudo para garantizar la trazabilidad.
- Limpieza del título y de la respuesta aceptada: HTML, URLs, duplicados y filtro mínimo de 100 palabras.
- Balanceo determinista a un máximo de 100 registros por categoría.
- Traducción al español del título y del texto, con un límite de 4.500 caracteres por contenido.
- Validación de la distribución, auditoría de muestras y exportación del dataset final en `procesados/`.


In [ ]:
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests
from deep_translator import GoogleTranslator
from tqdm.notebook import tqdm

In [ ]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / "data_science").exists() and (candidate / "README.md").exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / "data_science" / "data").resolve())
CARPETA_CRUDOS = str((project_root / "data_science" / "data" / "crudos").resolve())
CARPETA_PROCESADOS = str((project_root / "data_science" / "data" / "procesados").resolve())


print(f'📁 Proyecto local: {project_root.name}')
print(f'✅ Ruta de datos existe: {Path(CARPETA_DATA).exists()}')
print(f'✅ Ruta de crudos existe: {Path(CARPETA_CRUDOS).exists()}')
print(f"📁 Datos crudos: {Path(CARPETA_CRUDOS).relative_to(project_root)}")
print(f'✅ Ruta de procesados existe: {Path(CARPETA_PROCESADOS).exists()}')
print(f"📁 Datos procesados: {Path(CARPETA_PROCESADOS).relative_to(project_root)}")


## 2. Datos crudos — Extracción desde Stack Overflow

La primera versión de este trabajo surgió para reforzar **Mobile** y
**Frontend**. Después se decidió completar las siete categorías con el mismo
criterio: usar contenido técnico real obtenido desde la API pública de Stack
Exchange y conservar la respuesta aceptada de cada pregunta, porque su formato
explicativo se aproxima más a documentación o a un tutorial.

Tags usados en la versión final:

- Mobile: android, ios, flutter, kotlin, swift
- Frontend: javascript, css, reactjs, vue.js, angular
- Backend: java, spring, node.js, django, php
- Bases de Datos: sql, mysql, postgresql, mongodb, sql-server
- Cloud: amazon-web-services, azure, google-cloud-platform, docker, kubernetes
- Data Science: machine-learning, pandas, tensorflow, scikit-learn, numpy
- DevOps: jenkins, terraform, git, continuous-integration, ansible


### 2.1 Prueba de conexión con la API


In [ ]:
respuesta = requests.get(
    "https://api.stackexchange.com/2.3/questions",
    params={
        "tagged": "android",
        "site": "stackoverflow",
        "pagesize": 5
    }
)

print("Código de respuesta:", respuesta.status_code)
datos = respuesta.json()
print(datos["items"][0]["title"])


In [ ]:
def buscar_preguntas(tag, cantidad_paginas=1, pagesize=100):
    """
    Trae preguntas de Stack Overflow que tengan un tag específico (ej: 'android').
    Reintenta automáticamente si algo falla, hasta 3 veces por página.
    """
    todas_las_preguntas = []
    url = "https://api.stackexchange.com/2.3/questions"

    for pagina in range(1, cantidad_paginas + 1):
        parametros = {
            "tagged": tag,
            "site": "stackoverflow",
            "pagesize": pagesize,
            "page": pagina,
            "filter": "withbody",   # para que nos traiga el texto completo, no solo el título
            "sort": "votes",
            "order": "desc"
        }

        intentos = 0
        maximo_intentos = 3
        exito = False

        while intentos < maximo_intentos and not exito:
            try:
                respuesta = requests.get(url, params=parametros, timeout=10)

                if respuesta.status_code == 200:
                    datos = respuesta.json()
                    todas_las_preguntas.extend(datos.get("items", []))
                    exito = True
                    print(f"Tag '{tag}', página {pagina}: {len(datos.get('items', []))} preguntas traídas.")

                elif respuesta.status_code == 429:
                    print("La API dice que vamos muy rápido. Esperando 30 segundos...")
                    time.sleep(30)
                    intentos += 1

                else:
                    print(f"Código inesperado ({respuesta.status_code}). Reintentando en 5 segundos...")
                    intentos += 1
                    time.sleep(5)

            except requests.exceptions.RequestException as error:
                print(f"Problema de conexión: {error}. Reintentando en 5 segundos...")
                intentos += 1
                time.sleep(5)

        if not exito:
            print(f"No se pudo traer la página {pagina} del tag '{tag}' después de {maximo_intentos} intentos.")

        time.sleep(1)  # pausa chica entre páginas, para no saturar la API

    return todas_las_preguntas

In [ ]:
preguntas_android = buscar_preguntas(tag="android", cantidad_paginas=1, pagesize=20)
print("\nTotal de preguntas traídas:", len(preguntas_android))
print("Primer título:", preguntas_android[0]["title"])

In [ ]:
def convertir_a_dataframe(preguntas, tag_usado):
    filas = []
    for pregunta in preguntas:
        filas.append({
            "titulo": pregunta.get("title", ""),
            "pregunta_texto": pregunta.get("body", ""),
            "question_id": pregunta.get("question_id"),
            "accepted_answer_id": pregunta.get("accepted_answer_id"),
            "tag_original": tag_usado,
            "autor": pregunta.get("owner", {}).get("display_name", "Desconocido"),
            "url": pregunta.get("link", ""),
            "votos": pregunta.get("score", 0)
        })
    return pd.DataFrame(filas)

In [ ]:
df_android = convertir_a_dataframe(preguntas_android, "android")
df_android.head()

In [ ]:
preguntas_android = buscar_preguntas(tag="android", cantidad_paginas=2, pagesize=100)
print("\nTotal de preguntas traídas:", len(preguntas_android))

In [ ]:
df_android = convertir_a_dataframe(preguntas_android, "android")
print("Filas en la tabla:", len(df_android))
df_android.head()

In [ ]:
tags_por_categoria = {
    "Mobile": ["android", "ios", "flutter", "kotlin", "swift"],
    "Frontend": ["javascript", "css", "reactjs", "vue.js", "angular"],
    "Backend": ["java", "spring", "node.js", "django", "php"],
    "Bases de Datos": ["sql", "mysql", "postgresql", "mongodb", "sql-server"],
    "Cloud": ["amazon-web-services", "azure", "google-cloud-platform", "docker", "kubernetes"],
    "Data Science": ["machine-learning", "pandas", "tensorflow", "scikit-learn", "numpy"],
    "DevOps": ["jenkins", "terraform", "git", "continuous-integration", "ansible"]
}

In [ ]:
todos_los_dataframes = [df_android]  # ya tenemos Android, lo arrancamos con eso

for categoria, lista_tags in tags_por_categoria.items():
    for tag in lista_tags:
        if tag == "android":
            continue  # ya lo trajimos arriba, no lo repetimos

        print(f"\n--- Trayendo tag '{tag}' (categoría: {categoria}) ---")
        preguntas = buscar_preguntas(tag=tag, cantidad_paginas=2, pagesize=100)
        df_tag = convertir_a_dataframe(preguntas, tag)
        df_tag["categoria_equipo"] = categoria  # guardamos también la categoría del proyecto, no solo el tag
        todos_los_dataframes.append(df_tag)

print(f"\n✅ Listo. Se trajeron {len(todos_los_dataframes)} tablas en total.")

In [ ]:
df_completo = pd.concat(todos_los_dataframes, ignore_index=True)
print("Total de filas juntando todo:", len(df_completo))
df_completo["categoria_equipo"].value_counts()

In [ ]:
df_completo["categoria_equipo"] = df_completo["categoria_equipo"].fillna("Mobile")
df_completo["categoria_equipo"].value_counts()

In [ ]:
os.makedirs(CARPETA_CRUDOS, exist_ok=True)
os.makedirs(CARPETA_PROCESADOS, exist_ok=True)

print("Carpeta crudos:", os.path.exists(CARPETA_CRUDOS))
print("Carpeta procesados:", os.path.exists(CARPETA_PROCESADOS))


In [ ]:
ruta_completa_cruda = f'{CARPETA_CRUDOS}/stackexchange_corregido_crudo.csv'
df_completo.to_csv(ruta_completa_cruda, index=False)

print("Guardado en:", Path(ruta_completa_cruda).relative_to(project_root))
print("Total de filas:", len(df_completo))

In [ ]:
def buscar_respuestas(lista_ids_respuestas, tamano_lote=100):
    """
    Trae el contenido de varias respuestas de Stack Overflow a la vez,
    usando sus IDs. Se pide de a lotes (tamano_lote) porque la API
    no deja pedir una lista infinita de una sola vez.
    """
    respuestas_encontradas = {}
    url = "https://api.stackexchange.com/2.3/answers/{}"

    for inicio in range(0, len(lista_ids_respuestas), tamano_lote):
        lote = lista_ids_respuestas[inicio:inicio + tamano_lote]
        ids_texto = ";".join(str(int(id_)) for id_ in lote)

        parametros = {
            "site": "stackoverflow",
            "filter": "withbody",
            "pagesize": tamano_lote
        }

        intentos = 0
        maximo_intentos = 3
        exito = False

        while intentos < maximo_intentos and not exito:
            try:
                respuesta = requests.get(url.format(ids_texto), params=parametros, timeout=10)

                if respuesta.status_code == 200:
                    datos = respuesta.json()
                    for item in datos.get("items", []):
                        respuestas_encontradas[item["answer_id"]] = item.get("body", "")
                    exito = True
                    print(f"Lote {inicio // tamano_lote + 1}: {len(datos.get('items', []))} respuestas traídas.")

                elif respuesta.status_code == 429:
                    print("La API dice que vamos muy rápido. Esperando 30 segundos...")
                    time.sleep(30)
                    intentos += 1

                else:
                    print(f"Código inesperado ({respuesta.status_code}). Reintentando en 5 segundos...")
                    intentos += 1
                    time.sleep(5)

            except requests.exceptions.RequestException as error:
                print(f"Problema de conexión: {error}. Reintentando en 5 segundos...")
                intentos += 1
                time.sleep(5)

        time.sleep(1)

    return respuestas_encontradas

In [ ]:
ids_respuestas_validas = df_completo["accepted_answer_id"].dropna().tolist()
print("Cantidad de respuestas a traer:", len(ids_respuestas_validas))

mapa_respuestas = buscar_respuestas(ids_respuestas_validas)

In [ ]:
def buscar_texto_respuesta(id_respuesta):
    if pd.isna(id_respuesta):
        return None
    return mapa_respuestas.get(int(id_respuesta))

df_completo["texto"] = df_completo["accepted_answer_id"].apply(buscar_texto_respuesta)

print("Filas totales:", len(df_completo))
print("Filas con respuesta encontrada:", df_completo["texto"].notna().sum())

In [ ]:
df_completo = df_completo[df_completo["texto"].notna()].reset_index(drop=True)
print("Filas finales, con respuesta técnica real:", len(df_completo))

In [ ]:
ruta_completa_cruda = f'{CARPETA_CRUDOS}/stackexchange_corregido_crudo.csv'
df_completo.to_csv(ruta_completa_cruda, index=False)

print("Guardado en:", Path(ruta_completa_cruda).relative_to(project_root))
print("Total de filas:", len(df_completo))

In [ ]:
def limpiar_html_urls(texto):
    texto = re.sub(r'<[^>]+>', ' ', str(texto))
    texto = re.sub(r'http\S+', '', texto)
    return texto.strip()

df_completo["texto"] = df_completo["texto"].apply(limpiar_html_urls)
df_completo["titulo"] = df_completo["titulo"].apply(limpiar_html_urls)

df_completo[["titulo", "texto"]].head(3)

In [ ]:
antes = len(df_completo)
df_completo = df_completo.drop_duplicates(subset="texto").reset_index(drop=True)
print(f"Duplicados eliminados: {antes - len(df_completo)} (quedan {len(df_completo)})")

antes = len(df_completo)
df_completo = df_completo[df_completo["texto"].str.split().str.len() >= 100].reset_index(drop=True)
print(f"Textos con menos de 100 palabras eliminados: {antes - len(df_completo)} (quedan {len(df_completo)})")

In [ ]:
df_final = pd.DataFrame({
    "titulo": df_completo["titulo"],
    "texto": df_completo["texto"],
    "categoria": df_completo["categoria_equipo"],
    "autor": df_completo["autor"],
    "tema": df_completo["tag_original"],
    "tipo": "articulo"
})

print("Distribución antes de balancear:")
print(df_final["categoria"].value_counts())

In [ ]:
TOPE_POR_CATEGORIA = 100  # el número que veníamos manejando; confirmalo con el equipo antes de la entrega final

df_final_balanceado = pd.concat([
    grupo.sample(min(len(grupo), TOPE_POR_CATEGORIA), random_state=42)
    for _, grupo in df_final.groupby("categoria")
]).reset_index(drop=True)

print("=== DISTRIBUCIÓN FINAL ===")
print(df_final_balanceado["categoria"].value_counts())

for categoria, cantidad in df_final_balanceado["categoria"].value_counts().items():
    if cantidad < 30:
        print(f"⚠️ ADVERTENCIA: '{categoria}' tiene {cantidad} registros (por debajo del mínimo de 30).")
    else:
        print(f"✅ '{categoria}': OK ({cantidad} registros)")

In [ ]:
ruta_final = f'{CARPETA_PROCESADOS}/dataset_FINAL_stackexchange_corregido.csv'
df_final_balanceado.to_csv(ruta_final, index=False)

print("\n✅Guardado en:", Path(ruta_final).relative_to(project_root))
print("Total de filas:", len(df_final_balanceado))

print("\n=== MUESTRA DE AUDITORÍA (10 filas al azar) ===")
df_final_balanceado.sample(10, random_state=42)

In [ ]:
def traducir_texto(texto, max_caracteres=4500):
    if not texto or str(texto).strip() == "":
        return texto
    texto_recortado = str(texto)[:max_caracteres]
    try:
        return GoogleTranslator(source="en", target="es").translate(texto_recortado)
    except Exception as error:
        print(f"No se pudo traducir una fila: {error}")
        return texto

In [ ]:
prueba = df_final_balanceado["texto"].iloc[0]
print("ORIGINAL:", prueba[:200])
print("\nTRADUCIDO:", traducir_texto(prueba)[:200])

In [ ]:
tqdm.pandas()

print("Traduciendo títulos...")
df_final_balanceado["titulo"] = df_final_balanceado["titulo"].progress_apply(traducir_texto)

print("Traduciendo textos (esto va a tardar más, son textos más largos)...")
df_final_balanceado["texto"] = df_final_balanceado["texto"].progress_apply(traducir_texto)

print("\n✅ Traducción completa")

In [ ]:
df_final_balanceado = df_final_balanceado.drop(columns=["tema"])
df_final_balanceado.head(3)

In [ ]:
ruta_final_corregida = f'{CARPETA_PROCESADOS}/dataset_FINAL_stackexchange.csv'
df_final_balanceado.to_csv(ruta_final_corregida, index=False)

print("✅Guardado:", Path(ruta_final_corregida).relative_to(project_root))
print("Total de filas:", len(df_final_balanceado))
df_final_balanceado.head(3)